# 인공지능기초 7장 실습 · 합성곱 신경망(CNN)

**실습 목표** — 강의에서 손으로 계산한 것을 코드로 똑같이 재현하고, 그 원리가 실제 분류기에서 어떻게 작동하는지 확인한다.

| 순서 | 내용 | 강의 슬라이드 | 시간 |
|---|---|---|---|
| 0 | 환경 준비 · 텐서 모양 규칙 | 4 | 2분 |
| 1 | 합성곱 손계산을 코드로 검증 · 출력 크기 공식 | 8, 10 | 4분 |
| 2 | 필터는 특징 탐지기다: 엣지 필터 | 9 | 2.5분 |
| 3 | 풀링과 이동 불변성 | 13, 14 | 2.5분 |
| 4 | 데이터 준비 (Fashion-MNIST) | 19 | 1.5분 |
| 5 | CNN 구축 · 파라미터 수 추적 | 19, 20 | 3분 |
| 6 | 학습과 평가 | 19 | 2.5분 |
| 7 | MLP와 비교 · 이동 실험 | 5, 14 | 3.5분 |
| 8 | 학습된 필터와 특징 맵 보기 | 15 | 2분 |
| – | 정리 · 과제 7 안내 | – | 1.5분 |

> **시작 전에** Colab 메뉴 `런타임 → 런타임 유형 변경 → T4 GPU`를 선택하세요.
> 그래프의 제목·축 이름은 Colab에서 한글이 깨지지 않도록 영어로 적었습니다.

## 0. 환경 준비

In [3]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

keras.utils.set_random_seed(42)          # 실행할 때마다 같은 결과가 나오도록
print("TensorFlow", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU") or "없음 (CPU로도 실행은 됩니다)")

ModuleNotFoundError: No module named 'distutils'

## 1. 합성곱 손계산을 코드로 검증 (슬라이드 8)

강의 공식 그대로 구현합니다.

$$y[i,j] = \sum_m \sum_n x[i+m,\, j+n]\cdot w[m,n] + b$$

필터를 뒤집지 않는다는 점(교차상관)에 주의하세요.

In [ ]:
def conv2d_naive(x, w, b=0, stride=1):
    """패딩 없는 2차원 합성곱(교차상관). 이중 for문으로 '밀면서 곱하고 더한다'."""
    N, F = x.shape[0], w.shape[0]
    O = (N - F) // stride + 1
    y = np.zeros((O, O))
    for i in range(O):
        for j in range(O):
            patch = x[i*stride:i*stride+F, j*stride:j*stride+F]   # 1) 겹치기
            y[i, j] = np.sum(patch * w) + b                         # 2) 곱하고 더하기
    return y                                                         # 3) 미끄러뜨리며 반복한 결과

x = np.array([[1, 0, 1, 2, 0],
              [0, 1, 2, 1, 1],
              [2, 1, 0, 1, 0],
              [1, 0, 1, 2, 1],
              [0, 2, 1, 0, 1]], dtype=np.float32)

w = np.array([[1, 0, -1],
              [1, 0, -1],
              [1, 0, -1]], dtype=np.float32)

y_hand = conv2d_naive(x, w)
print(y_hand)        # 슬라이드 8의 정답: [[0,-2,2],[0,-2,1],[1,0,0]]

같은 계산을 TensorFlow로 해 봅니다. TensorFlow의 텐서 모양은 **(배치, 높이, 너비, 채널)** 이고,
필터 모양은 **(F, F, 입력 채널, 출력 채널)** 입니다.

In [ ]:
x_tf = x.reshape(1, 5, 5, 1)       # (B, H, W, C)
w_tf = w.reshape(3, 3, 1, 1)       # (F, F, C_in, K)
y_tf = tf.nn.conv2d(x_tf, w_tf, strides=1, padding="VALID")

print(y_tf.numpy().squeeze())
print("손계산과 같은가?", np.allclose(y_hand, y_tf.numpy().squeeze()))

### 출력 크기 공식 확인 (슬라이드 10)

$$O = \left\lfloor \frac{N + 2P - F}{S} \right\rfloor + 1$$

공식으로 계산한 값과 Keras 층이 실제로 내놓는 크기를 비교합니다.
Keras의 `padding="same"`은 스트라이드가 1일 때 출력 크기를 유지하도록 패딩을 자동으로 넣고,
그 밖의 패딩 값은 `ZeroPadding2D`로 직접 붙입니다.

In [ ]:
def out_size(N, F, P, S):
    return (N + 2*P - F) // S + 1

cases = [  # (N, F, P, S, 설명)
    (5,   3, 0, 1, "hand example"),
    (5,   3, 1, 1, "same padding"),
    (32,  5, 0, 1, "LeNet-5 C1"),
    (224, 7, 3, 2, "ResNet first layer"),
]
for N, F, P, S, name in cases:
    inp = keras.Input(shape=(N, N, 1))
    h = layers.ZeroPadding2D(P)(inp) if P > 0 else inp
    h = layers.Conv2D(1, F, strides=S, padding="valid")(h)
    print(f"{name:20s} N={N:3d} F={F} P={P} S={S}  공식={out_size(N,F,P,S):3d}  Keras={h.shape[1]:3d}")

## 2. 필터는 특징 탐지기다 (슬라이드 9)

먼저 슬라이드의 숫자 예제(`10 10 10 200 200 200`이 세 행 반복)를 그대로 재현합니다.

In [ ]:
row = np.array([10, 10, 10, 200, 200, 200], dtype=np.float32)
img = np.stack([row, row, row])                 # 3 × 6 이미지

v_edge = np.array([[1, 0, -1]] * 3, dtype=np.float32)              # 수직 엣지
h_edge = v_edge.T.copy()                                            # 수평 엣지: [[1,1,1],[0,0,0],[-1,-1,-1]]

out = tf.nn.conv2d(img.reshape(1, 3, 6, 1), v_edge.reshape(3, 3, 1, 1), 1, "VALID")
print(out.numpy().squeeze())    # 슬라이드 9: [0, -570, -570, 0]

이제 실제 의류 이미지에 같은 필터를 적용합니다. 데이터는 4절에서 본격적으로 다루므로 여기서는 한 장만 꺼내 씁니다.

In [ ]:
(x_train_raw, y_train), (x_test_raw, y_test) = keras.datasets.fashion_mnist.load_data()
class_names = ["T-shirt", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

sample = x_train_raw[0].astype(np.float32) / 255.0          # 0번 이미지: Ankle boot
s4 = sample.reshape(1, 28, 28, 1)
v_map = tf.nn.conv2d(s4, v_edge.reshape(3, 3, 1, 1), 1, "SAME").numpy().squeeze()
h_map = tf.nn.conv2d(s4, h_edge.reshape(3, 3, 1, 1), 1, "SAME").numpy().squeeze()

fig, ax = plt.subplots(1, 3, figsize=(10, 3.4))
ax[0].imshow(sample, cmap="gray");                         ax[0].set_title(f"input: {class_names[y_train[0]]}")
ax[1].imshow(v_map, cmap="bwr", vmin=-2, vmax=2);          ax[1].set_title("vertical-edge filter")
ax[2].imshow(h_map, cmap="bwr", vmin=-2, vmax=2);          ax[2].set_title("horizontal-edge filter")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

**관찰 포인트** — 평평한 영역(배경, 신발 안쪽)은 0(흰색)에 가깝고, 경계에서만 빨강(+)·파랑(−)으로 강하게 반응합니다.
수직 엣지 필터는 세로 경계를, 수평 엣지 필터는 신발 밑창 같은 가로 경계를 찾아냅니다.
CNN은 이런 필터를 사람이 정하지 않고 **역전파로 학습**합니다(8절에서 확인).

## 3. 풀링과 이동 불변성 (슬라이드 13, 14)

In [ ]:
fmap = np.array([[1, 3, 2, 1],
                 [4, 6, 5, 0],
                 [3, 1, 1, 2],
                 [0, 2, 7, 4]], dtype=np.float32).reshape(1, 4, 4, 1)

print("최대 풀링\n", layers.MaxPooling2D(2)(fmap).numpy().squeeze())       # [[6,5],[3,7]]
print("평균 풀링\n", layers.AveragePooling2D(2)(fmap).numpy().squeeze())   # [[3.5,2],[1.5,3.5]]
print("학습 파라미터 수:", layers.MaxPooling2D(2).count_params())

슬라이드 14의 1차원 예제입니다. 같은 특징(값 9)이 한 칸 이동했을 때 풀링 결과가 어떻게 되는지 봅니다.

In [ ]:
pool1d = layers.MaxPooling1D(2)
signals = {"A (reference)": [0, 9, 0, 0],
           "B (shift left 1)": [9, 0, 0, 0],
           "C (shift right 1)": [0, 0, 9, 0]}
for name, s in signals.items():
    out = pool1d(np.array(s, dtype=np.float32).reshape(1, 4, 1)).numpy().squeeze()
    print(f"{name:18s} {s} -> {out}")

B는 A와 같은 창(앞의 두 칸) 안에서 움직였기 때문에 결과가 같고, C는 창 경계를 넘었기 때문에 결과가 달라집니다.
즉 풀링이 주는 불변성은 **국소적·근사적**입니다. 7절에서 실제 이미지로 다시 확인합니다.

## 4. 데이터 준비 (슬라이드 4, 19)

Fashion-MNIST는 28×28 흑백 의류 이미지 10종, 훈련 6만 장·테스트 1만 장입니다.
Keras의 `Conv2D`는 **(B, H, W, C)** 모양을 기대하므로 흑백 이미지에도 채널 축(C=1)을 붙여야 합니다.

In [ ]:
x_train = (x_train_raw.astype("float32") / 255.0)[..., np.newaxis]   # (60000, 28, 28, 1)
x_test  = (x_test_raw.astype("float32")  / 255.0)[..., np.newaxis]   # (10000, 28, 28, 1)
print("x_train:", x_train.shape, " y_train:", y_train.shape)
print("x_test :", x_test.shape,  " y_test :", y_test.shape)

fig, ax = plt.subplots(2, 5, figsize=(10, 4.4))
for c, a in enumerate(ax.flat):
    idx = np.where(y_train == c)[0][0]
    a.imshow(x_train[idx].squeeze(), cmap="gray"); a.set_title(class_names[c]); a.axis("off")
plt.tight_layout(); plt.show()

## 5. CNN 구축 · 파라미터 수 추적 (슬라이드 19, 20)

슬라이드 19의 모델을 그대로 만듭니다. 마지막 층은 소프트맥스 없이 **로짓**을 내고,
손실 함수에서 `from_logits=True`로 소프트맥스를 함께 처리합니다.

In [ ]:
def build_cnn():
    return keras.Sequential([
        layers.Input(shape=(28, 28, 1)),
        layers.Conv2D(32, 3, padding="same", activation="relu"),   # 28×28×32
        layers.MaxPooling2D(2),                                     # 14×14×32
        layers.Conv2D(64, 3, padding="same", activation="relu"),   # 14×14×64
        layers.MaxPooling2D(2),                                     # 7×7×64
        layers.Flatten(),                                           # 3136
        layers.Dense(128, activation="relu"),
        layers.Dense(10),                                           # logits
    ], name="cnn")

cnn = build_cnn()
cnn.summary()

`summary()`의 Output Shape는 `(None, 28, 28, 32)`처럼 **채널이 마지막**에 옵니다(None = 배치 크기).
슬라이드 20 표의 `(B, 32, 28, 28)`은 채널을 앞에 두는 PyTorch 표기이고, 파라미터 수는 두 프레임워크가 같습니다.

공식 $(F\cdot F\cdot C_{in}+1)\cdot K$ 로 직접 계산해 summary와 맞춰 봅시다.

In [ ]:
manual = {
    "conv 1→32":     (3*3*1  + 1) * 32,
    "conv 32→64":    (3*3*32 + 1) * 64,
    "dense 3136→128": (7*7*64 + 1) * 128,
    "dense 128→10":  (128 + 1) * 10,
}
for k, v in manual.items():
    print(f"{k:16s} {v:>9,}")
total = sum(manual.values())
print(f"{'합계':15s} {total:>9,}   (summary: {cnn.count_params():,})")
print(f"완전연결층 3136→128의 비중: {manual['dense 3136→128']/total:.1%}")

## 6. 학습과 평가 (슬라이드 19)

학습 설정: `SparseCategoricalCrossentropy(from_logits=True)`, `Adam(1e-3)`, 배치 64, 5 에폭.
훈련 데이터의 10%는 검증용으로 떼어 둡니다. GPU 런타임에서 1분 안팎이 걸립니다.

In [ ]:
def compile_model(model):
    model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                  metrics=["accuracy"])
    return model

compile_model(cnn)
hist_cnn = cnn.fit(x_train, y_train, batch_size=64, epochs=5, validation_split=0.1, verbose=2)

In [ ]:
def plot_history(hist, title):
    h = hist.history
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
    ax[0].plot(h["loss"], "o-", label="train"); ax[0].plot(h["val_loss"], "o-", label="val")
    ax[0].set_title(f"{title} loss"); ax[0].set_xlabel("epoch"); ax[0].legend()
    ax[1].plot(h["accuracy"], "o-", label="train"); ax[1].plot(h["val_accuracy"], "o-", label="val")
    ax[1].set_title(f"{title} accuracy"); ax[1].set_xlabel("epoch"); ax[1].legend()
    plt.tight_layout(); plt.show()

plot_history(hist_cnn, "CNN")
test_loss, test_acc = cnn.evaluate(x_test, y_test, verbose=0)
print(f"CNN 테스트 정확도: {test_acc:.4f}")

In [ ]:
# 무엇을 틀렸는지 보기: 테스트 이미지 중 오답 8장
pred = cnn.predict(x_test, verbose=0).argmax(axis=1)
wrong = np.where(pred != y_test)[0][:8]
fig, ax = plt.subplots(1, 8, figsize=(14, 2.4))
for a, i in zip(ax, wrong):
    a.imshow(x_test[i].squeeze(), cmap="gray"); a.axis("off")
    a.set_title(f"T:{class_names[y_test[i]]}\nP:{class_names[pred[i]]}", fontsize=8)
plt.tight_layout(); plt.show()

## 7. MLP와 비교 · 이동 실험 (슬라이드 5, 14)

같은 학습 설정으로 MLP(Flatten → Dense 128 → Dense 10)를 학습합니다.

In [ ]:
mlp = keras.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dense(10),
], name="mlp")
compile_model(mlp)
hist_mlp = mlp.fit(x_train, y_train, batch_size=64, epochs=5, validation_split=0.1, verbose=2)
mlp_acc = mlp.evaluate(x_test, y_test, verbose=0)[1]

print(f"\n{'model':6s}{'params':>10s}{'test acc':>10s}")
print(f"{'MLP':6s}{mlp.count_params():>10,}{mlp_acc:>10.4f}")
print(f"{'CNN':6s}{cnn.count_params():>10,}{test_acc:>10.4f}")

**생각해 볼 점** — 이 CNN은 MLP보다 파라미터가 오히려 많습니다. 그런데 합성곱 두 층은 합쳐서 2만 개도 되지 않습니다.
파라미터를 늘린 범인은 합성곱이 아니라 `Flatten` 뒤의 완전연결층입니다. 이 문제는 **과제**에서 다룹니다.

이제 테스트 이미지를 오른쪽으로 0~4픽셀 옮겨 가며 두 모델의 정확도를 측정합니다. 빈 자리는 0(검정)으로 채웁니다.

In [ ]:
def shift_images(x, dx):
    """이미지를 오른쪽으로 dx 픽셀 이동하고 빈 칸은 0으로 채운다."""
    if dx == 0:
        return x
    out = np.zeros_like(x)
    out[:, :, dx:, :] = x[:, :, :-dx, :]
    return out

shifts = [0, 1, 2, 3, 4]
acc = {"MLP": [], "CNN": []}
for dx in shifts:
    xs = shift_images(x_test, dx)
    acc["MLP"].append(mlp.evaluate(xs, y_test, verbose=0)[1])
    acc["CNN"].append(cnn.evaluate(xs, y_test, verbose=0)[1])
    print(f"shift {dx}px   MLP {acc['MLP'][-1]:.4f}   CNN {acc['CNN'][-1]:.4f}")

plt.figure(figsize=(5.5, 3.6))
for k, v in acc.items():
    plt.plot(shifts, v, "o-", label=k)
plt.xlabel("shift (pixels)"); plt.ylabel("test accuracy"); plt.title("Accuracy under translation")
plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.show()

**관찰 포인트** — 두 모델 모두 정확도가 떨어지지만 보통 MLP가 훨씬 가파르게 떨어집니다.
CNN도 완전히 불변은 아닙니다. 합성곱(동변성)과 풀링(국소적 불변성)을 두 번 쌓았을 뿐이고,
마지막 `Flatten + Dense`는 다시 위치에 민감하기 때문입니다(슬라이드 14: “불변성은 국소적·근사적”).

## 8. 학습된 필터와 특징 맵 보기 (슬라이드 15)

첫 번째 합성곱 층의 필터 32개(3×3)와, 이미지 한 장이 각 합성곱 층을 통과한 결과(특징 맵)를 봅니다.

In [ ]:
w1, b1 = cnn.layers[0].get_weights()          # w1: (3, 3, 1, 32)
fig, ax = plt.subplots(4, 8, figsize=(9, 4.8))
for k, a in enumerate(ax.flat):
    a.imshow(w1[:, :, 0, k], cmap="bwr", vmin=-abs(w1).max(), vmax=abs(w1).max()); a.axis("off")
plt.suptitle("Learned 3x3 filters of conv layer 1"); plt.tight_layout(); plt.show()

In [ ]:
# 합성곱 층의 출력을 꺼내는 보조 모델
conv_layers = [l for l in cnn.layers if isinstance(l, layers.Conv2D)]
probe = keras.Model(inputs=cnn.inputs, outputs=[l.output for l in conv_layers])
f1, f2 = probe.predict(x_test[:1], verbose=0)
print("conv1 feature maps:", f1.shape, "  conv2 feature maps:", f2.shape)

fig, ax = plt.subplots(2, 9, figsize=(14, 3.6))
ax[0, 0].imshow(x_test[0].squeeze(), cmap="gray"); ax[0, 0].set_title("input")
ax[1, 0].axis("off")
for k in range(8):
    ax[0, k+1].imshow(f1[0, :, :, k], cmap="viridis"); ax[0, k+1].set_title(f"conv1 #{k}", fontsize=8)
    ax[1, k+1].imshow(f2[0, :, :, k], cmap="viridis"); ax[1, k+1].set_title(f"conv2 #{k}", fontsize=8)
for a in ax.flat: a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

**관찰 포인트** — conv1(28×28)의 특징 맵은 아직 원래 모양이 그대로 보이고 윤곽·밝기 대비에 반응합니다.
conv2(14×14)는 한 번 풀링을 거쳐 해상도가 낮고, 여러 엣지를 조합한 더 추상적인 패턴에 반응합니다.
깊어질수록 **공간 크기는 줄고 채널 수는 늘어나는** 구조를 눈으로 확인할 수 있습니다.

---

## 정리

1. 손계산한 합성곱·출력 크기·풀링 결과가 코드와 정확히 일치했다.
2. 파라미터 수 $(F\cdot F\cdot C_{in}+1)\cdot K$ 는 이미지 크기와 무관하고, 이 모델 파라미터의 95%는 완전연결층에 있다.
3. CNN은 MLP보다 이동에 강하지만 완전히 불변은 아니다.
4. 필터는 사람이 정하지 않고 데이터로부터 학습된다.

➡️ **과제 7**: `AI기초_07장_과제.ipynb`를 열어 진행하세요.